---
# Introducción

In [41]:
import pandas as pd
import numpy as np
from pathlib import Path
import sys

BASE_DIR = Path().resolve().parent
sys.path.append(str(BASE_DIR / 'scripts'))

In [42]:
df = pd.read_csv(BASE_DIR / 'data' / 'datos_procesados.csv')
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 25 columns):
 #   Column                     Non-Null Count  Dtype
---  ------                     --------------  -----
 0   linea_credito              30000 non-null  int64
 1   genero                     30000 non-null  int64
 2   estado_civil               30000 non-null  int64
 3   edad                       30000 non-null  int64
 4   comportamiento_septiembre  30000 non-null  int64
 5   comportamiento_agosto      30000 non-null  int64
 6   comportamiento_julio       30000 non-null  int64
 7   comportamiento_junio       30000 non-null  int64
 8   comportamiento_mayo        30000 non-null  int64
 9   comportamiento_abril       30000 non-null  int64
 10  estado_cuenta_septiembre   30000 non-null  int64
 11  estado_cuenta_agosto       30000 non-null  int64
 12  estado_cuenta_julio        30000 non-null  int64
 13  estado_cuenta_junio        30000 non-null  int64
 14  estado_cuenta_mayo         30000 

In [43]:
meses = ['abril', 'mayo', 'junio', 'julio', 'agosto', 'septiembre']

df_tendencias = pd.DataFrame({'mes': meses})

df_tendencias['tasa_utilizacion'] = [
    round(df[f'estado_cuenta_{mes}'].sum() / df['linea_credito'].sum(), 3)
    for mes in meses
]

df_tendencias['tasa_morosidad'] = [
    round((df[f'comportamiento_{mes}'] > 0).sum() / len(df), 3)
    for mes in meses
]

df_tendencias.head(10)

,mes,tasa_utilizacion,tasa_morosidad
0,abril,0.232,0.103
1,mayo,0.241,0.099
2,junio,0.258,0.117
3,julio,0.281,0.140
4,agosto,0.294,0.148
5,septiembre,0.306,0.227


In [44]:
meses_pares = [
    ('mayo', 'abril'),
    ('junio', 'mayo'),
    ('julio', 'junio'),
    ('agosto', 'julio'),
    ('septiembre', 'agosto')
]

tasa_recaudacion = [float('nan')]  # abril no tiene mes anterior

tasa_recaudacion += [
    round(df[f'abono_{mes}'].sum() / df[f'estado_cuenta_{mes_anterior}'].sum(), 3)
    for mes, mes_anterior in meses_pares
]

df_tendencias['tasa_recaudacion'] = tasa_recaudacion

df_tendencias['tasa_clientes_inactivos'] = [
    round((df[f'estado_cuenta_{mes}'] == 0).sum() / len(df), 3)
    for mes in meses
]

df_tendencias.head(10)

,mes,tasa_utilizacion,tasa_morosidad,tasa_recaudacion,tasa_clientes_inactivos
0,abril,0.232,0.103,NaN,0.134
1,mayo,0.241,0.099,0.123,0.117
2,junio,0.258,0.117,0.120,0.106
3,julio,0.281,0.140,0.121,0.096
4,agosto,0.294,0.148,0.126,0.084
5,septiembre,0.306,0.227,0.115,0.067


In [45]:
meses_pares = [
    ('mayo', 'abril'),
    ('junio', 'mayo'),
    ('julio', 'junio'),
    ('agosto', 'julio'),
    ('septiembre', 'agosto')
]

tasa_pago_total = [float('nan')]  # abril no tiene mes anterior

for mes, mes_anterior in meses_pares:
    # Condición 1: pago anticipado mismo mes
    cond1 = (df[f'abono_{mes}'] == df[f'estado_cuenta_{mes}']) & (df[f'abono_{mes}'] > 0)
    
    # Condición 2: pago normal mes desfasado
    cond2 = df[f'abono_{mes}'] >= df[f'estado_cuenta_{mes_anterior}']
    
    # Unión sin doble conteo
    clientes_pago_total = (cond1 | cond2).sum()
    
    tasa_pago_total.append(round(clientes_pago_total / len(df), 3))

df_tendencias['tasa_pago_total'] = tasa_pago_total

df_tendencias.head(10)

,mes,tasa_utilizacion,tasa_morosidad,tasa_recaudacion,tasa_clientes_inactivos,tasa_pago_total
0,abril,0.232,0.103,NaN,0.134,NaN
1,mayo,0.241,0.099,0.123,0.117,0.372
2,junio,0.258,0.117,0.120,0.106,0.347
3,julio,0.281,0.140,0.121,0.096,0.344
4,agosto,0.294,0.148,0.126,0.084,0.345
5,septiembre,0.306,0.227,0.115,0.067,0.333


In [46]:
df_tendencias['tasa_alto_riesgo'] = [
    round(
        (
            (df[f'estado_cuenta_{mes}'] / df['linea_credito'] > 0.90) &
            (df[f'abono_{mes}'] / df[f'estado_cuenta_{mes}'].replace(0, float('nan')) < 0.10)
        ).sum() / len(df), 3
    )
    for mes in meses
]

df_tendencias.head(10)

,mes,tasa_utilizacion,tasa_morosidad,tasa_recaudacion,tasa_clientes_inactivos,tasa_pago_total,tasa_alto_riesgo
0,abril,0.232,0.103,NaN,0.134,NaN,0.089
1,mayo,0.241,0.099,0.123,0.117,0.372,0.094
2,junio,0.258,0.117,0.120,0.106,0.347,0.122
3,julio,0.281,0.140,0.121,0.096,0.344,0.159
4,agosto,0.294,0.148,0.126,0.084,0.345,0.176
5,septiembre,0.306,0.227,0.115,0.067,0.333,0.189


In [47]:
tasa_impago = [float('nan')]  # abril sin mes anterior

for mes, mes_anterior in meses_pares:
    cond = (
        (df[f'estado_cuenta_{mes_anterior}'] > 0) &
        (df[f'abono_{mes}'] == 0)
    )
    tasa_impago.append(round(cond.sum() / len(df), 3))

df_tendencias['tasa_impago'] = tasa_impago

df_tendencias['orden'] = range(4, 10)

df_tendencias.head(10)

,mes,tasa_utilizacion,tasa_morosidad,tasa_recaudacion,tasa_clientes_inactivos,tasa_pago_total,tasa_alto_riesgo,tasa_impago,orden
0,abril,0.232,0.103,NaN,0.134,NaN,0.089,NaN,4
1,mayo,0.241,0.099,0.123,0.117,0.372,0.094,0.072,5
2,junio,0.258,0.117,0.120,0.106,0.347,0.122,0.081,6
3,julio,0.281,0.140,0.121,0.096,0.344,0.159,0.076,7
4,agosto,0.294,0.148,0.126,0.084,0.345,0.176,0.068,8
5,septiembre,0.306,0.227,0.115,0.067,0.333,0.189,0.074,9


In [48]:
df_tendencias.info()

<class 'pandas.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 9 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   mes                      6 non-null      str    
 1   tasa_utilizacion         6 non-null      float64
 2   tasa_morosidad           6 non-null      float64
 3   tasa_recaudacion         5 non-null      float64
 4   tasa_clientes_inactivos  6 non-null      float64
 5   tasa_pago_total          5 non-null      float64
 6   tasa_alto_riesgo         6 non-null      float64
 7   tasa_impago              5 non-null      float64
 8   orden                    6 non-null      int64  
dtypes: float64(7), int64(1), str(1)
memory usage: 564.0 bytes


In [49]:
df_tendencias = df_tendencias.rename(columns={
    'orden': 'orden',
    'mes': 'mes',
    'tasa_utilizacion': 'utilizacion',
    'tasa_morosidad': 'morosidad',
    'tasa_recaudacion': 'recaudacion',
    'tasa_clientes_inactivos': 'clientes_inactivos',
    'tasa_pago_total': 'pago_total',
    'tasa_alto_riesgo': 'alto_riesgo',
    'tasa_impago': 'impago'
})

df = df_tendencias

In [50]:
# Mantenemos 'mes' y 'orden' fijos, lo demás se estira hacia abajo
df = df.melt(id_vars=['mes', 'orden'], 
                   var_name='indicador', 
                   value_name='tasa')

In [51]:
df.head(20)

,mes,orden,indicador,tasa
0,abril,4,utilizacion,0.232
1,mayo,5,utilizacion,0.241
2,junio,6,utilizacion,0.258
3,julio,7,utilizacion,0.281
4,agosto,8,utilizacion,0.294
5,septiembre,9,utilizacion,0.306
6,abril,4,morosidad,0.103
7,mayo,5,morosidad,0.099
8,junio,6,morosidad,0.117
9,julio,7,morosidad,0.140


In [52]:
df.to_csv('tendencias_credito.csv', index=False)